<a href="https://colab.research.google.com/github/Fatou-Kine3/github_task/blob/main/SimpleConv2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Problem 1

In [1]:
import numpy as np

class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        """
        Initialize Conv2d layer

        Args:
            in_channels: Number of input channels (C_in)
            out_channels: Number of output channels (C_out)
            kernel_size: Tuple (Fh, Fw) or int for square kernel
            stride: Tuple (Sh, Sw) or int for square stride
            padding: Tuple (Ph, Pw) or int for square padding
        """
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # He initialization for weights
        self.W = np.random.randn(out_channels, in_channels, kernel_size[0], kernel_size[1]) * np.sqrt(2.0 / (in_channels * kernel_size[0] * kernel_size[1]))
        self.b = np.zeros(out_channels)

        # Cache for backward pass
        self.X_pad = None
        self.input_shape = None

    def forward(self, X):
        """
        Forward pass of Conv2d

        Args:
            X: Input of shape (N, C_in, H, W)

        Returns:
            Output of shape (N, C_out, H_out, W_out)
        """
        self.input_shape = X.shape
        N, C_in, H, W = X.shape

        # Pad input
        Ph, Pw = self.padding
        self.X_pad = np.pad(X, ((0,0), (0,0), (Ph, Ph), (Pw, Pw)), mode='constant')

        Sh, Sw = self.stride
        Fh, Fw = self.kernel_size

        # Calculate output dimensions
        H_out = (H + 2*Ph - Fh) // Sh + 1
        W_out = (W + 2*Pw - Fw) // Sw + 1

        # Initialize output
        out = np.zeros((N, self.out_channels, H_out, W_out))

        # Perform convolution
        for n in range(N):
            for m in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        # Extract patch
                        h_start = i * Sh
                        w_start = j * Sw
                        patch = self.X_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw]
                        # Apply convolution
                        out[n, m, i, j] = np.sum(patch * self.W[m]) + self.b[m]

        return out

    def backward(self, dout):
        """
        Backward pass of Conv2d

        Args:
            dout: Gradient from next layer of shape (N, C_out, H_out, W_out)

        Returns:
            dX: Gradient wrt input of shape (N, C_in, H, W)
        """
        N, C_out, H_out, W_out = dout.shape
        N, C_in, H, W = self.input_shape
        Sh, Sw = self.stride
        Fh, Fw = self.kernel_size
        Ph, Pw = self.padding

        # Initialize gradients
        dW = np.zeros_like(self.W)
        db = np.sum(dout, axis=(0, 2, 3))
        dX_pad = np.zeros_like(self.X_pad)

        # Compute gradients
        for n in range(N):
            for m in range(C_out):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw
                        patch = self.X_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw]
                        dW[m] += patch * dout[n, m, i, j]
                        dX_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw] += self.W[m] * dout[n, m, i, j]

        # Remove padding from dX
        if Ph == 0 and Pw == 0:
            dX = dX_pad
        else:
            dX = dX_pad[:, :, Ph:-Ph, Pw:-Pw] if Ph > 0 else dX_pad[:, :, :, Pw:-Pw]

        # Store gradients for optimizer
        self.grad_W = dW
        self.grad_b = db

        return dX

In [ ]:
## Probelm 2

In [3]:
import numpy as np

class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        """
        Initialize Conv2d layer

        Args:
            in_channels: Number of input channels (C_in)
            out_channels: Number of output channels (C_out)
            kernel_size: Tuple (Fh, Fw) or int for square kernel
            stride: Tuple (Sh, Sw) or int for square stride
            padding: Tuple (Ph, Pw) or int for square padding
        """
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # He initialization for weights (float64)
        self.W = np.random.randn(out_channels, in_channels, kernel_size[0], kernel_size[1]).astype(np.float64) * np.sqrt(2.0 / (in_channels * kernel_size[0] * kernel_size[1]))
        self.b = np.zeros(out_channels, dtype=np.float64)

        # Cache for backward pass
        self.X_pad = None
        self.input_shape = None
        self.Z = None  # Store output for ReLU gradient

    def forward(self, X):
        """
        Forward pass of Conv2d

        Args:
            X: Input of shape (N, C_in, H, W)

        Returns:
            Output of shape (N, C_out, H_out, W_out)
        """
        self.input_shape = X.shape
        N, C_in, H, W = X.shape

        # Pad input
        Ph, Pw = self.padding
        self.X_pad = np.pad(X, ((0,0), (0,0), (Ph, Ph), (Pw, Pw)), mode='constant')

        Sh, Sw = self.stride
        Fh, Fw = self.kernel_size

        # Calculate output dimensions
        H_out = (H + 2*Ph - Fh) // Sh + 1
        W_out = (W + 2*Pw - Fw) // Sw + 1

        # Initialize output
        out = np.zeros((N, self.out_channels, H_out, W_out), dtype=np.float64)

        # Perform convolution
        for n in range(N):
            for m in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        # Extract patch
                        h_start = i * Sh
                        w_start = j * Sw
                        patch = self.X_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw]
                        # Apply convolution
                        out[n, m, i, j] = np.sum(patch * self.W[m]) + self.b[m]

        self.Z = out  # Store for gradient computation
        return out

    def backward(self, dout):
        """
        Backward pass of Conv2d

        Args:
            dout: Gradient from next layer of shape (N, C_out, H_out, W_out)

        Returns:
            dX: Gradient wrt input of shape (N, C_in, H, W)
        """
        N, C_out, H_out, W_out = dout.shape
        N, C_in, H, W = self.input_shape
        Sh, Sw = self.stride
        Fh, Fw = self.kernel_size
        Ph, Pw = self.padding

        # Initialize gradients with float64 dtype
        dW = np.zeros_like(self.W, dtype=np.float64)
        db = np.sum(dout, axis=(0, 2, 3), dtype=np.float64)
        dX_pad = np.zeros_like(self.X_pad, dtype=np.float64)

        # Compute gradients
        for n in range(N):
            for m in range(C_out):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw
                        patch = self.X_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw]
                        dW[m] += patch * dout[n, m, i, j]
                        dX_pad[n, :, h_start:h_start+Fh, w_start:w_start+Fw] += self.W[m] * dout[n, m, i, j]

        # Remove padding from dX
        if Ph == 0 and Pw == 0:
            dX = dX_pad
        else:
            if Ph > 0 and Pw > 0:
                dX = dX_pad[:, :, Ph:-Ph, Pw:-Pw]
            elif Ph > 0:
                dX = dX_pad[:, :, Ph:-Ph, :]
            elif Pw > 0:
                dX = dX_pad[:, :, :, Pw:-Pw]
            else:
                dX = dX_pad

        # Store gradients for optimizer
        self.grad_W = dW
        self.grad_b = db

        return dX

In [ ]:
## Problem 3

In [4]:
def calculate_output_shape(H_in, W_in, kernel_size, stride=1, padding=0):
    """
    Calculate output shape for a 2D convolution

    Args:
        H_in: Input height
        W_in: Input width
        kernel_size: Tuple (Fh, Fw) or int
        stride: Tuple (Sh, Sw) or int
        padding: Tuple (Ph, Pw) or int

    Returns:
        Tuple (H_out, W_out)
    """
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)

    Fh, Fw = kernel_size
    Sh, Sw = stride
    Ph, Pw = padding

    # Reject nonpositive strides
    if Sh <= 0 or Sw <= 0:
        raise ValueError("Stride must be positive")

    # Reject filters larger than padded input
    if Fh > H_in + 2*Ph:
        raise ValueError(f"Filter height {Fh} larger than padded input height {H_in + 2*Ph}")
    if Fw > W_in + 2*Pw:
        raise ValueError(f"Filter width {Fw} larger than padded input width {W_in + 2*Pw}")

    H_out = (H_in + 2*Ph - Fh) // Sh + 1
    W_out = (W_in + 2*Pw - Fw) // Sw + 1

    return H_out, W_out

# Test
H_out, W_out = calculate_output_shape(6, 6, 3, 1, 0)
print(f"Output shape: ({H_out}, {W_out})")
assert (H_out, W_out) == (4, 4), f"Expected (4,4), got ({H_out}, {W_out})"
print("✓ Output shape calculation passed!")

Output shape: (4, 4)
✓ Output shape calculation passed!


In [ ]:
## Problem 4

In [5]:
class MaxPool2d:
    def __init__(self, kernel_size, stride=None, padding=0):
        """
        Initialize MaxPool2d layer

        Args:
            kernel_size: Tuple (Fh, Fw) or int
            stride: Tuple (Sh, Sw) or int (default: same as kernel_size)
            padding: Tuple (Ph, Pw) or int
        """
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if stride is None:
            stride = kernel_size
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # Cache for backward pass
        self.X = None
        self.max_indices = None
        self.input_shape = None

    def forward(self, X):
        """
        Forward pass of MaxPool2d

        Args:
            X: Input of shape (N, C, H, W)

        Returns:
            Output of shape (N, C, H_out, W_out)
        """
        self.input_shape = X.shape
        N, C, H, W = X.shape

        # Pad input
        Ph, Pw = self.padding
        self.X = np.pad(X, ((0,0), (0,0), (Ph, Ph), (Pw, Pw)), mode='constant', constant_values=-np.inf)

        Fh, Fw = self.kernel_size
        Sh, Sw = self.stride

        H_out = (H + 2*Ph - Fh) // Sh + 1
        W_out = (W + 2*Pw - Fw) // Sw + 1

        # Initialize output and max indices
        out = np.zeros((N, C, H_out, W_out))
        self.max_indices = np.zeros((N, C, H_out, W_out), dtype=np.int64)

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw
                        window = self.X[n, c, h_start:h_start+Fh, w_start:w_start+Fw]

                        # Find max value and its index
                        max_val = np.max(window)
                        max_idx = np.argmax(window)

                        out[n, c, i, j] = max_val
                        self.max_indices[n, c, i, j] = max_idx

        return out

    def backward(self, dout):
        """
        Backward pass of MaxPool2d

        Args:
            dout: Gradient from next layer of shape (N, C, H_out, W_out)

        Returns:
            dX: Gradient wrt input of shape (N, C, H, W)
        """
        N, C, H_out, W_out = dout.shape
        Ph, Pw = self.padding
        Fh, Fw = self.kernel_size
        Sh, Sw = self.stride

        # Initialize gradient with zeros
        dX_pad = np.zeros_like(self.X)

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw

                        # Get flat index and convert to 2D index
                        flat_idx = self.max_indices[n, c, i, j]
                        h_idx = h_start + flat_idx // Fw
                        w_idx = w_start + flat_idx % Fw

                        # Accumulate gradient at max position
                        dX_pad[n, c, h_idx, w_idx] += dout[n, c, i, j]

        # Remove padding
        if Ph == 0 and Pw == 0:
            dX = dX_pad
        else:
            dX = dX_pad[:, :, Ph:-Ph, Pw:-Pw] if Ph > 0 else dX_pad[:, :, :, Pw:-Pw]

        return dX

In [ ]:
## Problem 5

In [6]:
class AveragePool2d:
    def __init__(self, kernel_size, stride=None, padding=0):
        """
        Initialize AveragePool2d layer
        """
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if stride is None:
            stride = kernel_size
        if isinstance(stride, int):
            stride = (stride, stride)
        if isinstance(padding, int):
            padding = (padding, padding)

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # Cache for backward pass
        self.X = None
        self.input_shape = None

    def forward(self, X):
        """
        Forward pass of AveragePool2d
        """
        self.input_shape = X.shape
        N, C, H, W = X.shape

        Ph, Pw = self.padding
        self.X = np.pad(X, ((0,0), (0,0), (Ph, Ph), (Pw, Pw)), mode='constant')

        Fh, Fw = self.kernel_size
        Sh, Sw = self.stride

        H_out = (H + 2*Ph - Fh) // Sh + 1
        W_out = (W + 2*Pw - Fw) // Sw + 1

        out = np.zeros((N, C, H_out, W_out))

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw
                        window = self.X[n, c, h_start:h_start+Fh, w_start:w_start+Fw]
                        out[n, c, i, j] = np.mean(window)

        return out

    def backward(self, dout):
        """
        Backward pass of AveragePool2d
        """
        N, C, H_out, W_out = dout.shape
        Ph, Pw = self.padding
        Fh, Fw = self.kernel_size
        Sh, Sw = self.stride

        dX_pad = np.zeros_like(self.X)

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * Sh
                        w_start = j * Sw

                        # Distribute gradient equally across all elements in window
                        grad_val = dout[n, c, i, j] / (Fh * Fw)
                        dX_pad[n, c, h_start:h_start+Fh, w_start:w_start+Fw] += grad_val

        # Remove padding
        if Ph == 0 and Pw == 0:
            dX = dX_pad
        else:
            dX = dX_pad[:, :, Ph:-Ph, Pw:-Pw] if Ph > 0 else dX_pad[:, :, :, Pw:-Pw]

        return dX

In [ ]:
## Problem 6

In [7]:
class Flatten:
    def __init__(self):
        self.input_shape = None

    def forward(self, X):
        """
        Forward pass of Flatten layer

        Args:
            X: Input of shape (N, C, H, W)

        Returns:
            Output of shape (N, C*H*W)
        """
        self.input_shape = X.shape
        N = X.shape[0]
        return X.reshape(N, -1)

    def backward(self, dout):
        """
        Backward pass of Flatten layer

        Args:
            dout: Gradient from next layer of shape (N, C*H*W)

        Returns:
            dX: Gradient wrt input of shape (N, C, H, W)
        """
        return dout.reshape(self.input_shape)

In [ ]:
## Problem 7

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import time

class Dense:
    """Simple fully connected layer"""
    def __init__(self, input_size, output_size):
        self.W = np.random.randn(input_size, output_size).astype(np.float64) * np.sqrt(2.0 / input_size)
        self.b = np.zeros(output_size, dtype=np.float64)
        self.X = None
        self.Z = None

    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b
        return self.Z

    def backward(self, dout):
        self.grad_W = self.X.T @ dout
        self.grad_b = np.sum(dout, axis=0)
        dX = dout @ self.W.T
        return dX

class Scratch2dCNNClassifier:
    def __init__(self):
        self.conv1 = None
        self.pool1 = None
        self.conv2 = None
        self.pool2 = None
        self.flatten = None
        self.fc1 = None
        self.fc2 = None
        self.fc3 = None
        self.learning_rate = 0.01

    def softmax(self, X):
        exp_X = np.exp(X - np.max(X, axis=1, keepdims=True))
        return exp_X / np.sum(exp_X, axis=1, keepdims=True)

    def cross_entropy_loss(self, y_pred, y_true):
        N = y_pred.shape[0]
        return -np.sum(y_true * np.log(y_pred + 1e-10)) / N

    def forward(self, X, is_training=True):
        # First conv block
        X = self.conv1.forward(X)
        X = np.maximum(0, X)
        X = self.pool1.forward(X)

        # Second conv block
        X = self.conv2.forward(X)
        X = np.maximum(0, X)
        X = self.pool2.forward(X)

        # Flatten
        X = self.flatten.forward(X)

        # Fully connected layers
        X = self.fc1.forward(X)
        X = np.maximum(0, X)
        X = self.fc2.forward(X)
        X = np.maximum(0, X)
        X = self.fc3.forward(X)

        # Softmax
        X = self.softmax(X)
        return X

    def backward(self, dout):
        # FC3 backprop
        dout = self.fc3.backward(dout)
        dout = dout * (self.fc2.Z > 0)
        dout = self.fc2.backward(dout)
        dout = dout * (self.fc1.Z > 0)
        dout = self.fc1.backward(dout)

        # Flatten backprop
        dout = self.flatten.backward(dout)

        # Second conv block backprop
        dout = self.pool2.backward(dout)
        dout = dout * (self.conv2.Z > 0)
        dout = self.conv2.backward(dout)

        # First conv block backprop
        dout = self.pool1.backward(dout)
        dout = dout * (self.conv1.Z > 0)
        dout = self.conv1.backward(dout)

        return dout

    def update_parameters(self):
        for layer in [self.conv1, self.conv2, self.fc1, self.fc2, self.fc3]:
            layer.W -= self.learning_rate * layer.grad_W
            layer.b -= self.learning_rate * layer.grad_b

    def build_network(self):
        self.conv1 = Conv2d(1, 6, 5, 1, 0)
        self.pool1 = MaxPool2d(2, 2)
        self.conv2 = Conv2d(6, 16, 5, 1, 0)
        self.pool2 = MaxPool2d(2, 2)
        self.flatten = Flatten()
        self.fc1 = None
        self.fc2 = None
        self.fc3 = None

    def initialize_fc_layers(self, fc1_input_size):
        self.fc1 = Dense(fc1_input_size, 120)
        self.fc2 = Dense(120, 84)
        self.fc3 = Dense(84, 10)

    def train_step(self, X_batch, y_batch):
        y_pred = self.forward(X_batch, is_training=True)
        loss = self.cross_entropy_loss(y_pred, y_batch)
        dout = (y_pred - y_batch) / X_batch.shape[0]
        self.backward(dout)
        self.update_parameters()
        return loss

    def predict(self, X):
        y_pred = self.forward(X, is_training=False)
        return np.argmax(y_pred, axis=1)

    def accuracy(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == np.argmax(y, axis=1))

def train_mnist_fast():
    """
    Fast training on MNIST using subset of data
    """
    print("Loading MNIST dataset...")
    X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
    X = X.astype(np.float32) / 255.0
    y = y.astype(np.int32)

    # Use subset for faster training (10,000 samples)
    # Remove this line to train on full dataset
    X = X[:10000]
    y = y[:10000]

    X = X.reshape(-1, 1, 28, 28)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    y_train_onehot = np.eye(10)[y_train].astype(np.float64)
    y_test_onehot = np.eye(10)[y_test].astype(np.float64)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train_onehot, test_size=0.1, random_state=42)

    print(f"Training: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

    # Build model
    model = Scratch2dCNNClassifier()
    model.build_network()
    model.learning_rate = 0.01

    # Get FC input size
    sample = X_train[:1]
    conv1_out = model.conv1.forward(sample)
    conv1_out = np.maximum(0, conv1_out)
    pool1_out = model.pool1.forward(conv1_out)
    conv2_out = model.conv2.forward(pool1_out)
    conv2_out = np.maximum(0, conv2_out)
    pool2_out = model.pool2.forward(conv2_out)
    flat_out = model.flatten.forward(pool2_out)
    fc1_input_size = flat_out.shape[1]
    print(f"FC1 input size: {fc1_input_size}")
    model.initialize_fc_layers(fc1_input_size)

    # Training
    batch_size = 64
    epochs = 5  # Reduced epochs for speed

    train_losses = []
    val_losses = []
    val_accuracies = []

    print("\nStarting training (fast version)...")
    start_time = time.time()

    for epoch in range(epochs):
        indices = np.random.permutation(len(X_train))
        X_train_shuffled = X_train[indices]
        y_train_shuffled = y_train[indices]

        epoch_train_loss = 0
        num_batches = 0

        for i in range(0, len(X_train), batch_size):
            X_batch = X_train_shuffled[i:i+batch_size]
            y_batch = y_train_shuffled[i:i+batch_size]

            loss = model.train_step(X_batch, y_batch)
            epoch_train_loss += loss
            num_batches += 1

        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)

        # Validation
        val_pred = model.forward(X_val, is_training=False)
        val_loss = model.cross_entropy_loss(val_pred, y_val)
        val_losses.append(val_loss)

        val_acc = model.accuracy(X_val, y_val)
        val_accuracies.append(val_acc)

        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} [{elapsed:.1f}s] Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Test
    test_acc = model.accuracy(X_test, y_test_onehot)

    print(f"\n{'='*50}")
    print(f"PROBLEM 7 RESULTS")
    print(f"{'='*50}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Training time: {elapsed:.1f} seconds")

    # Plot
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss Curves')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(val_accuracies)
    plt.xlabel('Epoch')
    plt.ylabel('Validation Accuracy')
    plt.title('Validation Accuracy')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return model, test_acc

# Run Problem 7 (fast version)
if __name__ == "__main__":
    model, test_acc = train_mnist_fast()

Loading MNIST dataset...
Training: 7200, Validation: 800, Test: 2000
FC1 input size: 256

Starting training (fast version)...
Epoch 1/5 [552.1s] Train Loss: 1.5536, Val Loss: 0.9062, Val Acc: 0.7175
Epoch 2/5 [1103.1s] Train Loss: 0.6073, Val Loss: 0.5779, Val Acc: 0.8225
Epoch 3/5 [1652.1s] Train Loss: 0.3967, Val Loss: 0.5789, Val Acc: 0.8125
Epoch 4/5 [2199.5s] Train Loss: 0.3150, Val Loss: 0.4265, Val Acc: 0.8762


In [ ]:
## Problem 8

In [ ]:
def calculate_lenet_shapes():
    """
    Calculate intermediate shapes for LeNet architecture
    """
    print("\n" + "="*60)
    print("Problem 8: LeNet Architecture Shape Calculation")
    print("="*60)

    print(f"\nInput: (batch, 1, 28, 28)")

    # Conv1: 6 output channels, 5x5 filter, stride 1, no padding
    H1 = (28 + 0 - 5) // 1 + 1  # 24
    W1 = (28 + 0 - 5) // 1 + 1  # 24
    print(f"\n1. Conv1 (6 output channels, 5x5, stride 1)")
    print(f"   Output shape: (batch, 6, {H1}, {W1})")
    print(f"   Parameters: 6*1*5*5 + 6 = {6*1*5*5 + 6}")

    # ReLU + MaxPool1: 2x2, stride 2
    H2 = (H1 - 2) // 2 + 1  # 12
    W2 = (W1 - 2) // 2 + 1  # 12
    print(f"\n2. ReLU + MaxPool1 (2x2, stride 2)")
    print(f"   Output shape: (batch, 6, {H2}, {W2})")

    # Conv2: 16 output channels, 5x5 filter, stride 1, no padding
    H3 = (H2 + 0 - 5) // 1 + 1  # 8
    W3 = (W2 + 0 - 5) // 1 + 1  # 8
    print(f"\n3. Conv2 (16 output channels, 5x5, stride 1)")
    print(f"   Output shape: (batch, 16, {H3}, {W3})")
    print(f"   Parameters: 16*6*5*5 + 16 = {16*6*5*5 + 16}")

    # ReLU + MaxPool2: 2x2, stride 2
    H4 = (H3 - 2) // 2 + 1  # 4
    W4 = (W3 - 2) // 2 + 1  # 4
    print(f"\n4. ReLU + MaxPool2 (2x2, stride 2)")
    print(f"   Output shape: (batch, 16, {H4}, {W4})")

    # Flatten: (batch, 16*4*4) = (batch, 256)
    flatten_size = 16 * H4 * W4  # 256
    print(f"\n5. Flatten")
    print(f"   Output shape: (batch, {flatten_size})")

    # Fully connected layers
    print(f"\n6. FC1 (120 outputs)")
    print(f"   Output shape: (batch, 120)")
    print(f"   Parameters: {flatten_size}*120 + 120 = {flatten_size*120 + 120}")

    print(f"\n7. FC2 (84 outputs)")
    print(f"   Output shape: (batch, 84)")
    print(f"   Parameters: 120*84 + 84 = {120*84 + 84}")

    print(f"\n8. FC3 (10 outputs)")
    print(f"   Output shape: (batch, 10)")
    print(f"   Parameters: 84*10 + 10 = {84*10 + 10}")

    # Total parameters
    total_params = (6*1*5*5 + 6) + (16*6*5*5 + 16) + (flatten_size*120 + 120) + (120*84 + 84) + (84*10 + 10)
    print(f"\n{'='*60}")
    print(f"Total trainable parameters: {total_params:,}")
    print(f"{'='*60}")

    return flatten_size

# Run Problem 8
lenet_flatten_size = calculate_lenet_shapes()

In [ ]:
## Problem 9

In [1]:
def summarize_alexnet_vgg():
    """
    Summary of AlexNet and VGG16
    """
    print("\n" + "="*60)
    print("PROBLEM 9: Image Recognition Models Research")
    print("="*60)

    print("""

    ALEXNET (Krizhevsky et al., 2012)
    ================================

    Architecture:
    - 8 layers: 5 conv + 3 FC
    - Conv1: 96 × 11×11, stride 4
    - Conv2: 256 × 5×5, stride 1
    - Conv3: 384 × 3×3
    - Conv4: 384 × 3×3
    - Conv5: 256 × 3×3
    - FC: 4096 → 4096 → 1000

    Innovations:
    - ReLU activation (faster training)
    - Local Response Normalization
    - Dropout (50%)
    - Data augmentation
    - GPU training

    Historical Impact:
    - Won ImageNet 2012 (15.3% vs 26.2% error)
    - Revolutionized computer vision
    - Proved deep learning effectiveness


    VGG16 (Simonyan & Zisserman, 2014)
    =================================

    Architecture:
    - 16 layers: 13 conv + 3 FC
    - All conv: 3×3, stride 1
    - All pooling: 2×2, stride 2
    - Channel progression: 64→128→256→512→512
    - FC: 4096 → 4096 → 1000

    Key Features:
    - Depth: much deeper than AlexNet
    - Uniform 3×3 convolutions
    - Consistent design pattern

    Contribution:
    - Showed depth is important
    - Demonstrated 3×3 filter stacking
    - 7.3% top-5 error on ImageNet


    COMPARISON WITH LENET (Problem 8)
    ================================

    Feature        | LeNet      | AlexNet     | VGG16
    ---------------|------------|-------------|-------------
    Depth          | 5          | 8           | 16
    Parameters     | 60K        | 60M         | 138M
    Input Size     | 28×28      | 224×224     | 224×224
    Dataset        | MNIST      | ImageNet    | ImageNet
    Filter Sizes   | 5×5        | 11×11,5×5,3×3| 3×3
    Activation     | tanh/sigmoid| ReLU       | ReLU
    """)

# Run Problem 9
summarize_alexnet_vgg()


PROBLEM 9: Image Recognition Models Research

    
    ALEXNET (Krizhevsky et al., 2012)
    
    Architecture:
    - 8 layers: 5 conv + 3 FC
    - Conv1: 96 × 11×11, stride 4
    - Conv2: 256 × 5×5, stride 1
    - Conv3: 384 × 3×3
    - Conv4: 384 × 3×3
    - Conv5: 256 × 3×3
    - FC: 4096 → 4096 → 1000
    
    Innovations:
    - ReLU activation (faster training)
    - Local Response Normalization
    - Dropout (50%)
    - Data augmentation
    - GPU training
    
    Historical Impact:
    - Won ImageNet 2012 (15.3% vs 26.2% error)
    - Revolutionized computer vision
    - Proved deep learning effectiveness
    
    
    VGG16 (Simonyan & Zisserman, 2014)
    
    Architecture:
    - 16 layers: 13 conv + 3 FC
    - All conv: 3×3, stride 1
    - All pooling: 2×2, stride 2
    - Channel progression: 64→128→256→512→512
    - FC: 4096 → 4096 → 1000
    
    Key Features:
    - Depth: much deeper than AlexNet
    - Uniform 3×3 convolutions
    - Consistent design pattern
    
    Cont

In [ ]:
## Problem 10

In [2]:
def calculate_output_size_and_params():
    """
    Calculate output shape and parameter count
    """
    print("\n" + "="*60)
    print("PROBLEM 10: Output Size and Parameter Count")
    print("="*60)

    def conv_params(C_in, C_out, Fh, Fw, stride, padding, H_in, W_in):
        Sh, Sw = stride if isinstance(stride, tuple) else (stride, stride)
        Ph, Pw = padding if isinstance(padding, tuple) else (padding, padding)

        params = C_out * C_in * Fh * Fw + C_out
        H_out = (H_in + 2*Ph - Fh) // Sh + 1
        W_out = (W_in + 2*Pw - Fw) // Sw + 1

        return params, (H_out, W_out)

    print("\nCase 1:")
    print("3 channels, 144×144; 6 filters, 3×3; stride 1; no padding")
    params1, shape1 = conv_params(3, 6, 3, 3, 1, 0, 144, 144)
    print(f"Parameters: {params1:,}")
    print(f"Output shape: {shape1}")

    print("\nCase 2:")
    print("24 channels, 60×60; 48 filters, 3×3; stride 1; no padding")
    params2, shape2 = conv_params(24, 48, 3, 3, 1, 0, 60, 60)
    print(f"Parameters: {params2:,}")
    print(f"Output shape: {shape2}")

    print("\nCase 3:")
    print("10 channels, 20×20; 20 filters, 3×3; stride 2; no padding")
    params3, shape3 = conv_params(10, 20, 3, 3, 2, 0, 20, 20)
    print(f"Parameters: {params3:,}")
    print(f"Output shape: {shape3}")

    print("\n" + "="*60)
    print("EXPLANATION: Why floor division leaves unused edge")
    print("="*60)
    print("With stride 2 on 20×20 input with 3×3 filter:")
    print("Valid positions: 0, 2, 4, 6, 8, 10, 12, 14, 16, 18")
    print("Last window 18-20 covers positions 18, 19, 20")
    print("Remaining: positions 20-22 (beyond input)")
    print("Formula: (20 + 0 - 3) // 2 + 1 = 17//2 + 1 = 8 + 1 = 9")
    print("Edge positions are discarded because window doesn't fully fit")

# Run Problem 10
calculate_output_size_and_params()


PROBLEM 10: Output Size and Parameter Count

Case 1:
3 channels, 144×144; 6 filters, 3×3; stride 1; no padding
Parameters: 168
Output shape: (142, 142)

Case 2:
24 channels, 60×60; 48 filters, 3×3; stride 1; no padding
Parameters: 10,416
Output shape: (58, 58)

Case 3:
10 channels, 20×20; 20 filters, 3×3; stride 2; no padding
Parameters: 1,820
Output shape: (9, 9)

EXPLANATION: Why floor division leaves unused edge
With stride 2 on 20×20 input with 3×3 filter:
Valid positions: 0, 2, 4, 6, 8, 10, 12, 14, 16, 18
Last window 18-20 covers positions 18, 19, 20
Remaining: positions 20-22 (beyond input)
Formula: (20 + 0 - 3) // 2 + 1 = 17//2 + 1 = 8 + 1 = 9
Edge positions are discarded because window doesn't fully fit


In [ ]:
## Problem 11

In [3]:
def explain_filter_sizes():
    """
    Explanation of filter sizes
    """
    print("\n" + "="*60)
    print("PROBLEM 11: Research Filter Sizes")
    print("="*60)

    print("""

    1. WHY STACKED 3×3 FILTERS?
    ===========================

    Parameter Efficiency:
    - Two 3×3: 18 parameters (2 × 9)
    - One 5×5: 25 parameters
    - Three 3×3: 27 parameters
    - One 7×7: 49 parameters

    Same Receptive Field:
    - Two 3×3 (stride 1) = 5×5
    - Three 3×3 (stride 1) = 7×7

    Advantages:
    ✓ Fewer parameters (28% less than 5×5)
    ✓ More non-linearities (multiple ReLU)
    ✓ Better feature hierarchy
    ✓ Easier training
    ✓ Stronger regularization


    2. 1×1 CONVOLUTIONS
    ===================

    What They Do:
    - Change channels without combining spatial positions
    - Linear combination of input channels at each position

    Applications:
    a) Dimensionality Reduction
       - Reduce channels before 3×3 convs
       - Example: 256 → 64 → 256 (ResNet bottleneck)

    b) Feature Transformation
       - Learn linear combinations
       - Add non-linearity (ReLU after)

    c) Cross-channel Pooling
       - Pool information across channels
       - Similar to Network-in-Network


    3. COMPARISON
    =============

    Feature         | 5×5 Filter | Two 3×3 | 1×1 Conv
    ----------------|------------|---------|----------
    Receptive Field | 5×5        | 5×5     | 1×1
    Parameters      | 25C²       | 18C²    | C²
    Computation     | Higher     | Lower   | Lowest
    Non-linearities | 1          | 2       | 1
    Spatial Info    | Yes        | Yes     | No
    Channel Info    | Yes        | Yes     | Yes


    4. KEY PAPERS
    =============

    - VGG (2014): "Very Deep Convolutional Networks"
      → Demonstrated 3×3 filters are optimal

    - Inception (2014): "Going Deeper with Convolutions"
      → Introduced 1×1 for dimension reduction

    - ResNet (2015): "Deep Residual Learning"
      → Used 1×1 in bottleneck architecture
    """)

# Run Problem 11
explain_filter_sizes()


PROBLEM 11: Research Filter Sizes

    
    1. WHY STACKED 3×3 FILTERS?
    
    Parameter Efficiency:
    - Two 3×3: 18 parameters (2 × 9)
    - One 5×5: 25 parameters
    - Three 3×3: 27 parameters
    - One 7×7: 49 parameters
    
    Same Receptive Field:
    - Two 3×3 (stride 1) = 5×5
    - Three 3×3 (stride 1) = 7×7
    
    Advantages:
    ✓ Fewer parameters (28% less than 5×5)
    ✓ More non-linearities (multiple ReLU)
    ✓ Better feature hierarchy
    ✓ Easier training
    ✓ Stronger regularization
    
    
    2. 1×1 CONVOLUTIONS
    
    What They Do:
    - Change channels without combining spatial positions
    - Linear combination of input channels at each position
    
    Applications:
    a) Dimensionality Reduction
       - Reduce channels before 3×3 convs
       - Example: 256 → 64 → 256 (ResNet bottleneck)
    
    b) Feature Transformation
       - Learn linear combinations
       - Add non-linearity (ReLU after)
    
    c) Cross-channel Pooling
       - Pool in